### A. yfinance — Market Data Retrieval

**Purpose**

`yfinance` is a Python library that provides a convenient interface for downloading financial market data from Yahoo Finance. It is going to be useful when it comes to looking at stock prices and building the market-data portion of our backtesting system.

**Potential use cases**

* Retrieve historical open, high, low, close, and volume data.
* Examine adjusted prices, dividends, and stock splits.
* Build market features and calculate historical returns.
* Prototype and test trading strategies in a Jupyter Notebook.

**Advantages**

* Simple Python interface that integrates with pandas.
* Easy to understand and make adjsutments.
* Quick to prototype without manually constructing API requests.
* Useful for exploratory analysis and initial backtesting experiments.

**Disadvantages and limitations**
* It is an unofficial interface to Yahoo Finance, not an official Yahoo Finance market-data API contract.
* Availability, behavior, or data fields may change.
* Historical prices downloaded today do not automatically establish what price data was available at a particular historical time.
* Adjusted prices can incorporate corporate actions that occurred later, so the chosen price field and adjustment methodology must be documented.
* Downloaded data should be validated and cached rather than assumed to be permanently reproducible.

**Relevance to Quant Time Machine**

We should be using `yfinance` as way to test initial market-data experiments. Before we start building anything to do with point in type testing we need to investigate whether the data supports the historical availability and investement strategy testing to use. We also can not assume that downloading historical data will eliminate look-ahead bias.


In [ ]:
%pip install yfinance pandas matplotlib
## need to run this to be able to use yfinance

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Example: download daily data for Apple
ticker = yf.Ticker("AAPL")

prices = ticker.history(
    start="2023-01-01", #Start date 
    end="2025-01-01", 
    interval="1d",
    auto_adjust=False
)

display(prices.head())

In [ ]:
#Print statements to better understand the data

print("Rows and columns:", prices.shape)
print("\nColumns:")
print(prices.columns.tolist())

print("\nMissing values:")
print(prices.isna().sum())

print("\nDate range:")
print(prices.index.min(), "to", prices.index.max())

In [ ]:
# Use adjusted closing prices for this example.
# Document the adjustment methodology in any actual backtest.
prices["Daily_Return"] = prices["Adj Close"].pct_change()

display(prices[["Adj Close", "Daily_Return"]].tail())

prices["Daily_Return"].plot(
    title="AAPL Daily Returns",
    figsize=(12, 4)
)
plt.show()

> **Note (2026-10-11):** `Adj Close` is fine for a quick look at returns, but signals must never use adjusted price *levels*. An adjusted 2012 AAPL price already reflects the 2014 (7-for-1) and 2020 (4-for-1) splits, which nobody knew about in 2012. Our rule is to store raw prices plus split and dividend events and compute returns from them. See [`doc/time-model.md`](time-model.md) and the price-source comparison in [`doc/data-sources.md`](data-sources.md).

# FRED API — Economic Data Retrieval

### Purpose

FRED (Federal Reserve Economic Data), maintained by the Federal Reserve Bank of St. Louis, provides access to economic time series through an official API.

Examples include:

* Consumer Price Index (CPI) and inflation-related measures.
* Unemployment rates.
* Treasury yields and interest rates.
* GDP and other macroeconomic indicators.

**Advantages**

Provides structured economic time-series data.
Supports metadata and observation queries.
Allows filtering by observation dates and requesting different data frequencies.
Supports real-time-period parameters that help distinguish historical data from what is available today.

**Disadvantages and limitations**

* Economic data may be released after its observation date and revised afterward.
* Requesting a historical observation period does not necessarily retrieve the value known at that historical time.
* Indicators have different frequencies and release schedules, requiring careful alignment with market data.
* The API does not automatically prevent look-ahead bias in a backtest.
* C. ALFRED — Historical Economic Data Vintages

# ALFRED (Archival Federal Reserve Economic Data) preserves historical versions of economic series so researchers can examine what was reported at different points in time.

For example, an initial GDP estimate may later be revised. A backtest performed using today's revised value could use information that was not available when the original estimate was published.

**Advantages**

Helps reconstruct historical economic-data vintages.
Supports research into revisions and historical forecasting.
Is particularly useful for avoiding look-ahead bias caused by revised macroeconomic data.

**Disadvantages and limitations**

Requires understanding vintage dates and the distinction between observation dates and publication dates.
A vintage date alone may not precisely represent the time a release became available to a trading strategy.
Coverage and revision histories depend on the series.
The backtesting system must still enforce when each release became usable.

Relevance to Quant Time Machine

FRED is a convenient source for current historical economic series, while ALFRED is especially valuable when a strategy must use the economic-data versions that existed in the past. Both should be incorporated through a data layer that tracks observation dates, release availability, vintage dates, and source metadata.

Jeudy, E. (2026). Finance Fundamentals for Computer Science Students: A Learning Roadmap for Quant Time Machine. ChatGPT (GPT-6), OpenAI. Conversation dated October 5, 2026.

In [ ]:
%pip install fredapi pandas matplotlib

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from fredapi import Fred

# Never hardcode API keys. Put FRED_API_KEY in backend/.env (git-ignored; see backend/.env.example)
# or export it in your shell before starting Jupyter.
api_key = os.environ.get("FRED_API_KEY")
if not api_key:
    raise ValueError("Set the FRED_API_KEY environment variable first.")

fred = Fred(api_key=api_key)

In [ ]:
unemployment = fred.get_series(
    "UNRATE",
    observation_start="2018-01-01",
    observation_end="2024-12-31"
)

unemployment = unemployment.rename("Unemployment_Rate")

display(unemployment.head())
display(unemployment.tail())

In [ ]:
# What was known on 2020-06-01? Ask for the real-time period that contains that date.
# fredapi passes extra keyword arguments straight through to the FRED API.
as_of = "2020-06-01"

unemployment_vintage = fred.get_series(
    "UNRATE",
    observation_start="2018-01-01",
    observation_end="2020-06-01",
    realtime_start=as_of,
    realtime_end=as_of,
)

display(unemployment_vintage.tail(10))

> **Correction (2026-10-11):** the first version of the cell above passed `vintage_date=...`. That isn't a FRED API parameter, so FRED silently ignored it and returned *today's revised* values, including May and June 2020, which weren't published until June 5 and July 2, 2020. That is exactly the look-ahead bias this project exists to prevent.
>
> To see what was known on a date, set `realtime_start` and `realtime_end` to that date (as above), use the `vintage_dates` parameter, or use `output_type=4` for first releases. The verified behavior, including January 1990 unemployment as first published (5.3%) and as revised (5.4%), is in [`backend/spikes/alfred.py`](../backend/spikes/alfred.py) and [`doc/data-sources.md`](data-sources.md).

### Conclusion

For our initial Quant Time Machine prototype, we recommend evaluating `yfinance` for market-price retrieval and FRED/ALFRED for economic indicators and historical economic-data,

`yfinance` offers a simple way to experiment with market data, but its unofficial interface and historical-data limitations require testing before we can use it . FRED provides structured economic time series, while ALFRED adds important historical-vintage capabilities for research that must avoid look-ahead bias from revised economic data which our time machine is looking for.

Neither source alone solves the full point-in-time problem. Our architecture should preserve source metadata and retrieval records, model when information actually became available, and expose only eligible data to the backtesting engine.

The next step in investigating is seeing which of these API can help manage data the best

More API's to investigate 1. 
SEC EDGAR API,  Alpha Vantage, Nasdap Data Link

### Follow-up (2026-10-11)

The full comparison of data sources, their point-in-time fields, rate limits, and terms of use is in [`doc/data-sources.md`](data-sources.md), and the repeatable spike scripts are in [`backend/spikes/`](../backend/spikes/). Outputs were cleared from this notebook so it doesn't carry third-party data; re-run it locally to see them.